### Notebook to import results from CellReg and compare recording statistics across animals

*Last edit WMS 5/2/2026* -- only update is adding the heatmap plot

Written by WMS and Claude Sonnet 4.5/Opus 4.6; all code verified by WMS

Imports

In [ ]:
import h5py
import numpy as np
import os
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.patches import Patch

In [ ]:
#caiman imports if you want to confirm the index of imports to cellReg to those in original data structures produced after CNMF-E
#need to be in a conda env with caiman installed in order to do this (to make use of caiman's built-in contour finding function)

import caiman as cm
from caiman.utils.visualization import plot_contours, nb_view_patches, nb_plot_contour
from caiman.source_extraction.cnmf import cnmf as cnmf
from caiman.source_extraction.cnmf import params as params

In [ ]:
from session_config import *
from xsession_utils import *

Define paths and switches for saving outputs

In [ ]:
#define paths for saving
xsesh_path = r"C:\miniscope_files\analyzed\xsessionxanimal_results"
cellreg_res_path = os.path.join(xsesh_path,"cellreg_outputs")

In [ ]:
savefigs=False

In [ ]:
savecsvs=False

Load CellReg results

In [ ]:
#functions originally from MEK modified for use with WMS's file organization

def extract_cell_to_index_map_hdf5(mat_file_path):
    """Extract cell_to_index_map from MATLAB v7.3 (HDF5) file.
    Args:
        mat_file_path (str): Path to the .mat file.
    Returns:
        np.ndarray: Numpy array containing the cell_to_index_map.
    """
    with h5py.File(mat_file_path, 'r') as f:
        try:
            # Access the cell_to_index_map inside the structure
            cell_registered_struct = f['cell_registered_struct']
            index_map_ref = cell_registered_struct['cell_to_index_map']
            cell_to_index_map = np.array(index_map_ref)
            cell_index_map_reformat = np.squeeze(cell_to_index_map.T).astype('int')
            cell_index_map_df = pd.DataFrame(cell_index_map_reformat)
            return cell_index_map_df
        except KeyError as e:
            print(f"KeyError: {e} in {mat_file_path}")
            return None

#WMS slightly tweaked the below one
def extract_cell_to_index_map(animal_directories):
    """Extract the cell_to_index_map from each animal's .mat file.
    Args:
        animal_directories (dict): A dictionary containing directories for each animal.
    Returns:
        dict: A dictionary containing cell_to_index_map data for each animal and session.
    """
    cell_to_index_maps = {}
    for animal, directory in animal_directories.items():
        #cell_to_index_maps[animal] = {}
        if not os.path.exists(directory):
            print(f"Directory not found: {directory}")
            continue
        # Find the .mat file starting with "cellRegistered"
        mat_files = [f for f in os.listdir(directory) if f.startswith("cellRegistered") and f.endswith(".mat")]
        if not mat_files:
            print(f"No 'cellRegistered' file found in directory: {directory}")
            continue
        mat_file_path = os.path.join(directory, mat_files[0])
        try:
            # Extract the cell_to_index_map using the HDF5 reader
            cell_to_index_map = extract_cell_to_index_map_hdf5(mat_file_path)
            if cell_to_index_map is not None:
                cell_to_index_maps[animal] = cell_to_index_map
                #print(f"Successfully extracted cell_to_index_map for {animal}, {session_name}")
            else:
                print(f"Failed to extract cell_to_index_map for {animal}")
        except Exception as e:
            print(f"Error processing file: {mat_file_path}. Error: {e}")
    return cell_to_index_maps

In [ ]:
animal_ids = ['6175','7845','7981','8020','8689','8929','8946']
base_path = r"C:\miniscope_files\analyzed"

animal_directories = {
    animal_id : os.path.join(base_path, animal_id, f"CellReg") for animal_id in animal_ids
}

In [ ]:
cell_to_index_maps = extract_cell_to_index_map(animal_directories)

In [ ]:
cell_to_index_maps['8689']

In [ ]:
#convert to 0-indexed to match up better with the cell #s from CNMF-E outputs

def convert_to_zero_indexed(cell_index_map_df):
    """Convert MATLAB 1-indexed cell registration map to Python 0-indexed with NaNs.
    
    Args:
        cell_index_map_df (pd.DataFrame): DataFrame from extract_cell_to_index_map_hdf5
                                          with 1-indexed values where 0 = not detected
    
    Returns:
        pd.DataFrame: DataFrame with 0-indexed values and NaN for undetected cells
    """
    # Create a copy to avoid modifying the original
    df_converted = cell_index_map_df.copy()
    
    # Replace 0s with NaN (0 means cell not detected in that session)
    df_converted = df_converted.replace(0, np.nan)
    
    # Subtract 1 from all values to convert from 1-indexed to 0-indexed
    df_converted = df_converted - 1

    df_converted = df_converted.astype('Int64')
    
    return df_converted

In [ ]:
zero_index_maps = {}

for animal in animal_ids:
    new_map = convert_to_zero_indexed(cell_to_index_maps[animal])
    new_map.columns=SESSION_SUBSETS['social'] #relabel the columnns to reflect the true session indices
    zero_index_maps[animal]=new_map

In [ ]:
zero_index_maps['8689']

In [ ]:
#cell counts per session for each animal
cell_counts_per_session = {}
for animal, df in zero_index_maps.items():
    # Count non-NaN values in each column (session)
    cell_counts = df.notna().sum(axis=0)
    cell_counts_per_session[animal] = cell_counts
cell_counts_df = pd.DataFrame(cell_counts_per_session).T

In [ ]:
cell_counts_df

In [ ]:
if savecsvs:
    cell_counts_df.to_csv(os.path.join(cellreg_res_path,'num_neurons_per_session.csv'))

In [ ]:
#load in single cell data and check numcells against these values^

In [ ]:
all_sc_data = load_all_single_cell_data(ANIMAL_PATHS)

In [ ]:
#confirm that the sessions in the modified cellreg output refer to the same sessions in the single cell data results
for animal in animal_ids:
    for s_idx in range(NUM_SESSIONS):
        sc_data = all_sc_data.get(animal, {}).get(s_idx)
        if sc_data is None:
            continue
        expected = sc_data.get('numcells')
        cellreg_count = cell_counts_df.loc[animal, s_idx] if s_idx in cell_counts_df.columns else None
        if cellreg_count is None:
            continue
        # CellReg count should be = expected
        if cellreg_count == expected:
            print(f"{animal} session {s_idx}: {cellreg_count}/{expected} "
                  f"cells registered ({cellreg_count/expected*100:.0f}%)")
        else:
            print(f"======WARNING: {animal} session {s_idx}: {cellreg_count}/{expected} "
                  f"cells registered ({cellreg_count/expected*100:.0f}%)=====")

In [ ]:
#plot cell counts per session for each animal
plt.figure(figsize=(10, 6))
for animal in animal_ids:
    counts = cell_counts_df.loc[animal,:].values
    sessions = range(len(counts))  # Session numbers (1, 2, 3, ...)
    plt.plot(sessions, counts, marker='o', label=animal, linewidth=2)#color='gray'
plt.xlabel('Session', fontsize=12)
plt.ylabel('# neurons', fontsize=12)
plt.title('number of neurons detected per session (by animal)', fontsize=14)
plt.legend(title='Animal')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.xticks([0,1,2,3,4],SESSION_CONFIG['short_labels'][1:6])
if savefigs:
    plt.savefig(os.path.join(cellreg_res_path,"num_neurons_per_animal_per_session.svg"), dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
#calculate the average # of neurons recorded at each session across animals
mean_per_session = cell_counts_df.mean(axis=0)
std_per_session = cell_counts_df.std(axis=0)

print("Average # of neurons per session (across all animals):")
for session_idx in cell_counts_df.columns:
    print(f"Session {session_idx}: {mean_per_session[session_idx]:.1f} ± {std_per_session[session_idx]:.1f}")

In [ ]:
#calculate the average # of cells across all sessions for each animal
mean_per_animal = cell_counts_df.mean(axis=1)
std_per_animal = cell_counts_df.std(axis=1)

print("\nAverage cells per animal (across all sessions):")
for animal in mean_per_animal.index:
    print(f"{animal}: {mean_per_animal[animal]:.1f} ± {std_per_animal[animal]:.1f}")

In [ ]:
#now calculate the average # of neurons recorded in a given session across all animals and all sessions
all_counts = cell_counts_df.values.ravel()
mean_counts = all_counts.mean()
std_counts = all_counts.std(ddof=1)
print(f"Average # of neurons per session across all animals and all sessions: {mean_counts:.1f} ± {std_counts:.1f}")

In [ ]:
#now just get the total number of neurons for each animal and across all animals

total_cell_counts = {}
total_cell_counts_arr = []
for animal, df in zero_index_maps.items():
    num_cells = df.shape[0]
    total_cell_counts[animal]=num_cells
    total_cell_counts_arr.append(num_cells)

total_num_cells = sum(total_cell_counts_arr)

print(f"Total number of distinct cells across all animals and all sessions: {total_num_cells}")
total_cell_counts

In [ ]:
#testing the boolean logic for the next function
both_present = zero_index_maps['7981'].iloc[:,0].notna() & zero_index_maps['7981'].iloc[:, 1].notna()
both_present#.sum()

In [ ]:
def count_coregistered_consecutive_sessions(df):
    """Count neurons co-registered between each pair of consecutive sessions.
    
    Args:
        df (pd.DataFrame): Cell registration map with sessions as columns
    
    Returns:
        dict: Number of co-registered neurons for each consecutive pair
    """
    coregistered_counts = {}
    num_sessions = df.shape[1]
    
    for i in range(num_sessions - 1):
        # Check if neuron is present in BOTH session i and session i+1
        both_present = df.iloc[:, i].notna() & df.iloc[:, i+1].notna()
        count = both_present.sum()
        coregistered_counts[f'Session_{df.columns[i]}_to_{df.columns[i+1]}'] = count
    
    return coregistered_counts

# Calculate for each animal
coregistered_stats = {}
for animal, df in zero_index_maps.items():
    coregistered_stats[animal] = count_coregistered_consecutive_sessions(df)

# Print results
print("Neurons co-registered across consecutive sessions:\n")
for animal, counts in coregistered_stats.items():
    print(f"{animal}:")
    for pair, count in counts.items():
        print(f"  {pair}: {count} neurons")
    print()

# Create DataFrame for easier visualization
coregistered_df = pd.DataFrame(coregistered_stats).T
print("\nSummary DataFrame:")
coregistered_df

In [ ]:
if savecsvs:
    coregistered_df.to_csv(os.path.join(cellreg_res_path,'num_neurons_coregistered_per_session_pair.csv'))

In [ ]:
import matplotlib.pyplot as plt

# Plot co-registration across consecutive sessions
plt.figure(figsize=(10, 6))

for animal, counts in coregistered_stats.items():
    values = list(counts.values())
    plt.plot(range(len(values)), values, marker='o', label=animal, linewidth=2)

plt.xlabel('Session Pair', fontsize=12)
plt.ylabel('Number of Co-registered Neurons', fontsize=12)
plt.title('Neurons Co-registered Across Consecutive Sessions', fontsize=14)
plt.xticks(range(4), ['same sex novel<>familiar', 'same sex familiar<>opp. sex novel', 'opp. sex novel<>familiar', 'opp. sex pre<>post-separation'])
plt.legend(title='Animal')
plt.grid(True, alpha=0.3)
plt.tight_layout()

if savefigs:
    plt.savefig(os.path.join(cellreg_res_path,"num_neurons_coregistered_session_pairs.svg"), dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
# Average co-registration across animals
mean_coregistered = coregistered_df.mean(axis=0)
std_coregistered = coregistered_df.std(axis=0)

print("Average co-registration across animals:")
for pair in coregistered_df.columns:
    #pair_idx = pair.split('_')[1] + '-' + pair.split('_')[-1]
    print(f"{pair}: {mean_coregistered[pair]:.1f} ± {std_coregistered[pair]:.1f}")

In [ ]:
import matplotlib.pyplot as plt

def count_session1_neurons_over_time(df):
    """Count how many neurons from session 1 appear in each subsequent session.
    
    Args:
        df (pd.DataFrame): Cell registration map with sessions as columns
    
    Returns:
        dict: Number of session 1 neurons present in each session
    """
    # Find all neurons detected in session 1
    session1_neurons = df.iloc[:, 0].notna()
    
    # Count how many of those neurons appear in each session
    counts = {}
    num_sessions = df.shape[1]
    
    for session in range(num_sessions):
        # Of the neurons in session 1, how many are also in this session?
        present_in_both = session1_neurons & df.iloc[:, session].notna()
        counts[f'Session_{df.columns[session]}'] = present_in_both.sum()
    
    return counts

# Calculate for each animal
session1_tracking = {}
for animal, df in zero_index_maps.items():
    session1_tracking[animal] = count_session1_neurons_over_time(df)

# Print results
print("Session 1 neurons tracked across sessions:\n")
for animal, counts in session1_tracking.items():
    print(f"{animal}:")
    for session, count in counts.items():
        print(f"  {session}: {count} neurons")
    print()

# Create the plot
plt.figure(figsize=(10, 6))

for animal, counts in session1_tracking.items():
    sessions = range(len(counts))
    values = list(counts.values())
    plt.plot(sessions, values, marker='o', label=animal, linewidth=2)

plt.xlabel('Session', fontsize=12)
plt.ylabel('Number of SS-N Neurons Present', fontsize=12)
plt.title('Registration persistence of SS-N neurons across sessions', fontsize=14)
plt.xticks(range(5), SESSION_CONFIG['short_labels'][1:6])
plt.legend(title='Animal')
plt.grid(True, alpha=0.3)
plt.tight_layout()

if savefigs:
    plt.savefig(os.path.join(cellreg_res_path,"num_starting_neurons_persisting_across_sessions.svg"), dpi=300, bbox_inches='tight')

plt.show()

In [ ]:
#verifying that the cell in the given spatial_footprints' first dimension index is indeed the cell in caiman's A estimates
#note that you need to be running this notebook in the caiman conda env to run this section (may not be compatible with some of the cells above depending on what your installs are)

#i.e, this checks whether the inputs to cellreg (and thus their 1-indexed equivalents in cellReg's outputs) correspond to the original components in CNMF-e == the row order in Cdata.csv

In [ ]:
def plot_footprints_A_vs_mat(A, dims, spatial_footprints, highlight_neuron=None,
                               thr=0.5, thr_method='max',
                               outline_color='gray', highlight_color='black',
                               outline_linewidth=0.8, figsize=(14, 7),
                               save_path=None, show=True):
    """
    Side-by-side plot of spatial footprints from (1) CNMF-E estimates.A and 
    (2) a pre-exported spatial_footprints.mat file, both with the same neuron 
    optionally highlighted.
    
    Useful for verifying that the exported .mat file matches the source A matrix
    in both dimensional layout and neuron ordering.
    
    Parameters
    ----------
    A : sparse matrix
        Spatial components matrix from CNMF-E (estimates.A), shape (pixels × K).
    dims : tuple
        (height, width) of the FOV.
    spatial_footprints : np.ndarray
        Array of shape (n_neurons, height, width) loaded from a spatial_footprints.mat file.
    highlight_neuron : int or None
        0-indexed neuron to fill in highlight_color in both subplots. 
        If None, all neurons are shown as outlines only.
    thr : float
        Threshold for contour extraction.
    thr_method : str
        'max' or 'nrg'.
    outline_color : str
        Color for non-highlighted neuron outlines.
    highlight_color : str
        Fill color for the highlighted neuron.
    outline_linewidth : float
        Line width for outlines.
    figsize : tuple
    save_path : str or None
    show : bool
    
    Returns
    -------
    fig, (ax_A, ax_mat)
    """
    from skimage.measure import find_contours
    
    # --- Subplot 1: contours from A via caiman's get_contours ---
    contours_A = cm.utils.visualization.get_contours(A, dims, thr=thr, thr_method=thr_method)
    
    fig, (ax_A, ax_mat) = plt.subplots(1, 2, figsize=figsize)
    
    for i, d in enumerate(contours_A):
        coordinates = d['coordinates']
        x = coordinates[:, 0]
        y = coordinates[:, 1]
        
        if highlight_neuron is not None and i == highlight_neuron:
            ax_A.fill(x, y, facecolor=highlight_color, edgecolor=highlight_color,
                      linewidth=outline_linewidth, zorder=10)
        else:
            ax_A.plot(x, y, color=outline_color, linewidth=outline_linewidth, zorder=1)
    
    ax_A.axis('off')
    ax_A.invert_yaxis()
    ax_A.set_aspect('equal')
    ax_A.set_title('From cnm.estimates.A', fontsize=12)
    
    # --- Subplot 2: contours from spatial_footprints.mat ---
    # Each footprint is a 2D array; we threshold and trace contours directly
    n_neurons_sf = spatial_footprints.shape[0]
    
    for i in range(n_neurons_sf):
        footprint = spatial_footprints[i, :, :]
        
        # Apply the same thresholding as get_contours does internally
        if thr_method == 'max':
            if footprint.max() == 0:
                continue
            normalized = footprint / footprint.max()
            threshold_val = thr
        else:  # 'nrg'
            # Energy-based thresholding
            flat = footprint.ravel()
            nonzero = flat[flat > 0]
            if len(nonzero) == 0:
                continue
            sorted_vals = np.sort(nonzero)[::-1]
            cum_energy = np.cumsum(sorted_vals ** 2)
            cum_energy /= cum_energy[-1]
            normalized = np.ones_like(footprint, dtype=float)
            # This is an approximation — for exact match with get_contours
            # you'd replicate the full energy-based logic
            normalized = footprint / footprint.max() if footprint.max() > 0 else footprint
            threshold_val = thr
        
        # Find contours at the threshold
        vertices_list = find_contours(normalized.T, threshold_val)
        
        # Concatenate disjoint contours with nan separators (same convention as get_contours)
        all_coords = np.atleast_2d([np.nan, np.nan])
        for vtx in vertices_list:
            all_coords = np.concatenate(
                (all_coords, vtx, np.atleast_2d([np.nan, np.nan])), axis=0
            )
        
        if len(all_coords) <= 1:
            continue
        
        x = all_coords[:, 0]
        y = all_coords[:, 1]
        
        if highlight_neuron is not None and i == highlight_neuron:
            ax_mat.fill(x, y, facecolor=highlight_color, edgecolor=highlight_color,
                        linewidth=outline_linewidth, zorder=10)
        else:
            ax_mat.plot(x, y, color=outline_color, linewidth=outline_linewidth, zorder=1)
    
    ax_mat.axis('off')
    ax_mat.invert_yaxis()
    ax_mat.set_aspect('equal')
    ax_mat.set_title('From spatial_footprints.mat', fontsize=12)
    
    if highlight_neuron is not None:
        fig.suptitle(f'Neuron {highlight_neuron} highlighted', fontsize=13, y=1.02)
    
    fig.set_facecolor('white')
    plt.tight_layout()
    
    if save_path:
        plt.savefig(save_path, dpi=300, bbox_inches='tight')
    
    if show:
        plt.show()
    else:
        plt.close()
    
    return fig, (ax_A, ax_mat)

In [ ]:
from scipy.io import loadmat

def load_spatial_footprints_mat(path):
    mat_data = loadmat(path)
    return mat_data['spatial_footprints']

spatial_footprints = load_spatial_footprints_mat(
    r"C:\miniscope_files\analyzed\8020\CellReg\250425\spatial_footprints.mat"
)

A, dims = load_footprints(r"C:\miniscope_files\analyzed\8020\2025_04_25\16_12_33\250425_os_postsep")

plot_footprints_A_vs_mat(A, dims, spatial_footprints, highlight_neuron=17)

In [ ]:
#plotting code that will assess whether the indeces of each spatial_footprints.mat file's N dimension is really what is mapping onto the value in each row for the outputted CellReg results structure

In [ ]:
import scipy.io as sio
from matplotlib.patches import Polygon
from matplotlib.collections import PatchCollection

In [ ]:
def load_spatial_footprints(session_directories):
    """Load spatial footprints from all sessions.
    
    Args:
        session_directories (list): List of paths to directories containing spatial_footprints.mat
    
    Returns:
        list: List of spatial footprint arrays for each session
    """
    footprints_list = []
    for directory in session_directories:
        footprint_path = os.path.join(directory, 'spatial_footprints.mat')
        if not os.path.exists(footprint_path):
            print(f"Warning: {footprint_path} not found")
            footprints_list.append(None)
            continue
        
        data = sio.loadmat(footprint_path)
        footprints = data['spatial_footprints']  # Shape: (N_neurons, height, width)
        footprints_list.append(footprints)
        print(f"  {os.path.basename(directory)}: {footprints.shape[0]} neurons") #sanity check to make sure the footprints files were listed in the correct order -- x-reference to the expected # of neurons
        
    return footprints_list



In [ ]:
def get_neuron_contour(footprint, threshold=0.01):
    """Extract contour from a neuron's spatial footprint.
    
    Args:
        footprint (np.ndarray): 2D array representing neuron's spatial footprint
        threshold (float): Threshold for binarizing the footprint
    
    Returns:
        np.ndarray: Contour points (Nx2 array of [x, y] coordinates)
    """
    from skimage import measure
    
    # Binarize the footprint
    binary = footprint > threshold
    
    # Find contours
    contours = measure.find_contours(binary, 0.5)
    
    if len(contours) > 0:
        # Return the longest contour (likeliest to be the main cell body)
        return max(contours, key=len)
    else:
        return None

In [ ]:
def plot_tracked_neuron(cell_index_map_row, footprints_list, session_indices=None,
                         threshold=0.01, outline_color='gray', outline_alpha=0.3,
                         outline_linewidth=0.5, highlight_color='red',
                         highlight_alpha=0.5, highlight_linewidth=2,
                         global_neuron_id=None, figsize=None,
                         save_path=None, show=True):
    """
    Plot the same biologically-registered neuron across all sessions, with all 
    other neurons shown as faint outlines for spatial context.
    
    Parameters
    ----------
    cell_index_map_row : pd.Series
        One row from a zero_index_maps[animal] DataFrame. Index labels should 
        be the session indices (post-relabeling), values are 0-indexed neuron 
        IDs or NaN for sessions where the cell was not registered.
    footprints_list : list
        List of spatial footprint arrays (shape N × height × width) for each 
        session, in the same order as session_indices. Elements can be None 
        for sessions with no footprint file.
    session_indices : list[int] or None
        Which session indices the footprints_list corresponds to. Used to look 
        up short_labels from SESSION_CONFIG for subplot titles. If None, defaults 
        to [1, 2, 3, 4, 5] (the CellReg social sessions).
    threshold : float
        Threshold for get_neuron_contour() when tracing outlines.
    global_neuron_id : int or None
        The row index in the zero_index_maps DataFrame for this cell. Used only 
        for the figure title.
    figsize : tuple or None
        Defaults to (4 * n_sessions, 4) if not provided.
    
    Returns
    -------
    fig, axes
    """
    if session_indices is None:
        session_indices = SESSION_SUBSETS['social']
    
    n_sessions = len(footprints_list)
    if figsize is None:
        figsize = (4 * n_sessions, 4)
    
    fig, axes = plt.subplots(1, n_sessions, figsize=figsize)
    if n_sessions == 1:
        axes = [axes]
    
    for subplot_idx, (ax, footprints, s_idx) in enumerate(
            zip(axes, footprints_list, session_indices)):
        
        short_label = SESSION_CONFIG['short_labels'][s_idx]
        
        if footprints is None:
            ax.text(0.5, 0.5, 'No footprint data', ha='center', va='center',
                    transform=ax.transAxes, fontsize=10, color='gray')
            ax.set_title(short_label, fontsize=11)
            ax.axis('off')
            continue
        
        img_height, img_width = footprints.shape[1], footprints.shape[2]
        ax.set_xlim(0, img_width)
        ax.set_ylim(img_height, 0)
        ax.set_aspect('equal')
        
        # Draw all neurons as faint gray outlines
        n_neurons_this_session = footprints.shape[0]
        for n in range(n_neurons_this_session):
            contour = get_neuron_contour(footprints[n], threshold=threshold)
            if contour is not None:
                ax.plot(contour[:, 1], contour[:, 0],
                        color=outline_color, linewidth=outline_linewidth,
                        alpha=outline_alpha)
        
        # Look up this cell's neuron ID in this session
        neuron_id_this_session = cell_index_map_row[s_idx]
        
        if pd.isna(neuron_id_this_session):
            ax.set_title(f'{short_label}\nNot registered',
                         fontsize=10, color='gray')
        else:
            neuron_id_this_session = int(neuron_id_this_session)
            
            if neuron_id_this_session >= n_neurons_this_session:
                ax.set_title(f'{short_label}\nIndex out of range',
                             fontsize=10, color='orange')
            else:
                tracked_footprint = footprints[neuron_id_this_session]
                contour = get_neuron_contour(tracked_footprint, threshold=threshold)
                if contour is not None:
                    ax.fill(contour[:, 1], contour[:, 0],
                            color=highlight_color, alpha=highlight_alpha, zorder=10)
                    ax.plot(contour[:, 1], contour[:, 0],
                            color=highlight_color, linewidth=highlight_linewidth,
                            zorder=11)
                ax.set_title(f'{short_label}\nNeuron {neuron_id_this_session}',
                             fontsize=10, color=highlight_color, fontweight='bold')
        
        ax.axis('off')
    
    if global_neuron_id is not None:
        fig.suptitle(f'Registered cell #{global_neuron_id} across sessions',
                     fontsize=14, fontweight='bold')
    else:
        fig.suptitle('Registered cell across sessions', fontsize=14, fontweight='bold')
    
    plt.tight_layout()
    
    if save_path:
        plt.savefig(save_path, dpi=300, bbox_inches='tight')
    
    if show:
        plt.show()
    else:
        plt.close()
    
    return fig, axes

In [ ]:
#define all footprints in session idx order for each animal

footprints_lists_by_animal = {}

print('==8946==')
footprints_lists_by_animal['8946']=load_spatial_footprints([
    r"C:\miniscope_files\analyzed\8946\CellReg\251029",
    r"C:\miniscope_files\analyzed\8946\CellReg\251030",
    r"C:\miniscope_files\analyzed\8946\CellReg\251105",
    r"C:\miniscope_files\analyzed\8946\CellReg\251106",
    r"C:\miniscope_files\analyzed\8946\CellReg\251120"
])

print('==8929==')
footprints_lists_by_animal['8929']=load_spatial_footprints([
    r"C:\miniscope_files\analyzed\8929\CellReg\251029",
    r"C:\miniscope_files\analyzed\8929\CellReg\251030",
    r"C:\miniscope_files\analyzed\8929\CellReg\251105",
    r"C:\miniscope_files\analyzed\8929\CellReg\251106",
    r"C:\miniscope_files\analyzed\8929\CellReg\251120"
])

print('==8689==')
footprints_lists_by_animal['8689'] = load_spatial_footprints([
    r"C:\miniscope_files\analyzed\8689\CellReg\250812",
    r"C:\miniscope_files\analyzed\8689\CellReg\250813",
    r"C:\miniscope_files\analyzed\8689\CellReg\250819",
    r"C:\miniscope_files\analyzed\8689\CellReg\250820",
    r"C:\miniscope_files\analyzed\8689\CellReg\250903"
])

print('==8020==')
footprints_lists_by_animal['8020'] = load_spatial_footprints([
    r"C:\miniscope_files\analyzed\8020\CellReg\250403",
    r"C:\miniscope_files\analyzed\8020\CellReg\250404",
    r"C:\miniscope_files\analyzed\8020\CellReg\250410",
    r"C:\miniscope_files\analyzed\8020\CellReg\250411",
    r"C:\miniscope_files\analyzed\8020\CellReg\250425"
])

print('==7981==')
footprints_lists_by_animal['7981'] = load_spatial_footprints([
    r"C:\miniscope_files\analyzed\7981\CellReg\250324",
    r"C:\miniscope_files\analyzed\7981\CellReg\250325",
    r"C:\miniscope_files\analyzed\7981\CellReg\250331",
    r"C:\miniscope_files\analyzed\7981\CellReg\250401",
    r"C:\miniscope_files\analyzed\7981\CellReg\250415"
])

print('==7845==')
footprints_lists_by_animal['7845'] = load_spatial_footprints([
    r"C:\miniscope_files\analyzed\7845\CellReg\250114",
    r"C:\miniscope_files\analyzed\7845\CellReg\250115",
    r"C:\miniscope_files\analyzed\7845\CellReg\250121",
    r"C:\miniscope_files\analyzed\7845\CellReg\250122",
    r"C:\miniscope_files\analyzed\7845\CellReg\250205"
])

print('==6175==')
footprints_lists_by_animal['6175'] = load_spatial_footprints([
    r"C:\miniscope_files\analyzed\6175\CellReg\240207",
    r"C:\miniscope_files\analyzed\6175\CellReg\240208",
    r"C:\miniscope_files\analyzed\6175\CellReg\240214",
    r"C:\miniscope_files\analyzed\6175\CellReg\240215",
    r"C:\miniscope_files\analyzed\6175\CellReg\240229"
])

In [ ]:
# Pick a cell that's registered across all sessions to verify tracking
animal = '7981'
zero_map = zero_index_maps[animal]

# Find a row where the cell is present in all sessions
complete_rows = zero_map.dropna()
#global_id = complete_rows.index[14]  # uncomment if you want to check various fully-tracked cells

global_id = 3

# Load footprints for this animal's sessions (in order from session idx 1-5 as in SESSION_CONFIG)
footprints_list = footprints_lists_by_animal[animal]

fig, ax = plot_tracked_neuron(
    cell_index_map_row=zero_map.loc[global_id], footprints_list=footprints_list, session_indices=None, #this defaults to 1-5, the social sessions, which is what we want
    threshold=0.07, outline_color='black', outline_alpha=0.5, highlight_color='red', highlight_alpha=0.8, global_neuron_id=global_id
)

In [ ]:
#saving out neuron 19 from 8689 because it is a good example of a cell successfully registered across sessions
if savefigs:
    cellreg_success_save_path = os.path.join(cellreg_res_path,'8689_neuron_19_successful_cellReg.svg')
else:
    cellreg_success_save_path = None

In [ ]:
# Pick a cell that's registered across all sessions to verify tracking
animal = '8689'
zero_map = zero_index_maps[animal]

# Find a row where the cell is present in all sessions
complete_rows = zero_map.dropna()
#global_id = complete_rows.index[14]  # uncomment if you want to check various fully-tracked cells

global_id =19

# Load footprints for this animal's sessions (in order from session idx 1-5 as in SESSION_CONFIG)
footprints_list = footprints_lists_by_animal[animal]

fig, ax = plot_tracked_neuron(
    cell_index_map_row=zero_map.loc[global_id], footprints_list=footprints_list, session_indices=None, #this defaults to 1-5, the social sessions, which is what we want
    threshold=0.07, outline_color='black', outline_alpha=0.5, highlight_color='red', highlight_alpha=0.8, global_neuron_id=global_id, save_path=cellreg_success_save_path
)

In [ ]:
# Pick a cell that's registered across all sessions to verify tracking
animal = '8020'
zero_map = zero_index_maps[animal]

# Find a row where the cell is present in all sessions
complete_rows = zero_map.dropna()
#global_id = complete_rows.index[14]  # uncomment if you want to check various fully-tracked cells

global_id = 2

# Load footprints for this animal's sessions (in order from session idx 1-5 as in SESSION_CONFIG)
footprints_list = footprints_lists_by_animal[animal]

fig, ax = plot_tracked_neuron(
    cell_index_map_row=zero_map.loc[global_id], footprints_list=footprints_list, session_indices=None, #this defaults to 1-5, the social sessions, which is what we want
    threshold=0.07, outline_color='black', outline_alpha=0.5, highlight_color='red', highlight_alpha=0.8, global_neuron_id=global_id
)

In [ ]:
# Pick a cell that's registered across all sessions to verify tracking
animal = '7845'
zero_map = zero_index_maps[animal]

# Find a row where the cell is present in all sessions
complete_rows = zero_map.dropna()
#global_id = complete_rows.index[14]  # uncomment if you want to check various fully-tracked cells

global_id = 24

# Load footprints for this animal's sessions (in order from session idx 1-5 as in SESSION_CONFIG)
footprints_list = footprints_lists_by_animal[animal]

fig, ax = plot_tracked_neuron(
    cell_index_map_row=zero_map.loc[global_id], footprints_list=footprints_list, session_indices=None, #this defaults to 1-5, the social sessions, which is what we want
    threshold=0.07, outline_color='black', outline_alpha=0.5, highlight_color='red', highlight_alpha=0.8, global_neuron_id=global_id
)

In [ ]:
# Pick a cell that's registered across all sessions to verify tracking
animal = '6175'
zero_map = zero_index_maps[animal]

# Find a row where the cell is present in all sessions
complete_rows = zero_map.dropna()
#global_id = complete_rows.index[14]  # uncomment if you want to check various fully-tracked cells

global_id = 24

# Load footprints for this animal's sessions (in order from session idx 1-5 as in SESSION_CONFIG)
footprints_list = footprints_lists_by_animal[animal]

fig, ax = plot_tracked_neuron(
    cell_index_map_row=zero_map.loc[global_id], footprints_list=footprints_list, session_indices=None, #this defaults to 1-5, the social sessions, which is what we want
    threshold=0.07, outline_color='black', outline_alpha=0.5, highlight_color='red', highlight_alpha=0.8, global_neuron_id=global_id
)

In [ ]:
# Pick a cell that's registered across all sessions to verify tracking
animal = '8929'
zero_map = zero_index_maps[animal]

# Find a row where the cell is present in all sessions
complete_rows = zero_map.dropna()
#global_id = complete_rows.index[14]  # uncomment if you want to check various fully-tracked cells

global_id = 24

# Load footprints for this animal's sessions (in order from session idx 1-5 as in SESSION_CONFIG)
footprints_list = footprints_lists_by_animal[animal]

fig, ax = plot_tracked_neuron(
    cell_index_map_row=zero_map.loc[global_id], footprints_list=footprints_list, session_indices=None, #this defaults to 1-5, the social sessions, which is what we want
    threshold=0.07, outline_color='black', outline_alpha=0.5, highlight_color='red', highlight_alpha=0.8, global_neuron_id=global_id
)

In [ ]:
# Pick a cell that's registered across all sessions to verify tracking
animal = '8946'
zero_map = zero_index_maps[animal]

# Find a row where the cell is present in all sessions
complete_rows = zero_map.dropna()
#global_id = complete_rows.index[14]  # uncomment if you want to check various fully-tracked cells

global_id = 24

# Load footprints for this animal's sessions (in order from session idx 1-5 as in SESSION_CONFIG)
footprints_list = footprints_lists_by_animal[animal]

fig, ax = plot_tracked_neuron(
    cell_index_map_row=zero_map.loc[global_id], footprints_list=footprints_list, session_indices=None, #this defaults to 1-5, the social sessions, which is what we want
    threshold=0.07, outline_color='black', outline_alpha=0.5, highlight_color='red', highlight_alpha=0.8, global_neuron_id=global_id
)

In [ ]:
def color_footprints_by_tuning(animal_id, session_indices, footprints_list,
                                 all_sc_data, color_mixed_by='identity',
                                 threshold=0.05, figsize=None, fill_alpha=0.8,
                                 outline_color='black', outline_linewidth=0.5,
                                 outline_alpha=0.8, background_color=None,
                                 show=True, save_path=None):
    
    if session_indices is None:
        session_indices = SESSION_SUBSETS['social']
    
    n_sessions = len(footprints_list)
    if figsize is None:
        figsize = (4 * n_sessions, 4)
    
    fig, axes = plt.subplots(1, n_sessions, figsize=figsize)
    if n_sessions == 1:
        axes = [axes]
    
    def get_cell_color(footprint_idx, sc_data, mixed_sets):
        if footprint_idx in sc_data.get('partner_cells', []):
            return PARTNER_COLOR
        elif footprint_idx in sc_data.get('novel_cells', []):
            return NOVEL_COLOR
        elif footprint_idx in sc_data.get('left_cells', []):
            return LEFT_COLOR
        elif footprint_idx in sc_data.get('right_cells', []):
            return RIGHT_COLOR
        elif color_mixed_by == 'identity':
            if footprint_idx in mixed_sets['partner_mixed']:
                return PARTNER_MIXED_COLOR
            elif footprint_idx in mixed_sets['novel_mixed']:
                return NOVEL_MIXED_COLOR
        elif color_mixed_by == 'location':
            if footprint_idx in mixed_sets['left_mixed']:
                return LEFT_MIXED_COLOR
            elif footprint_idx in mixed_sets['right_mixed']:
                return RIGHT_MIXED_COLOR
        return '#D3D3D3'  # untuned or this session didn't assign tuning during the social trials
    
    for subplot_idx, (ax, footprints, s_idx) in enumerate(
            zip(axes, footprints_list, session_indices)):
        
        sc_data = all_sc_data.get(animal_id, {}).get(s_idx)
        short_label = SESSION_CONFIG['short_labels'][s_idx]
        
        if footprints is None or sc_data is None:
            msg = 'No footprint data' if footprints is None else 'No tuning data'
            ax.text(0.5, 0.5, msg, ha='center', va='center',
                    transform=ax.transAxes, fontsize=10, color='gray')
            ax.set_title(short_label, fontsize=11)
            ax.axis('off')
            continue

        num_cells_footprints = footprints.shape[0]
        num_cells_sc_data = sc_data['numcells']
        assert num_cells_footprints == num_cells_sc_data, (
            f"Cell count mismatch for animal {animal_id}, session {s_idx}: "
            f"footprints has {num_cells_footprints} cells but sc_data reports "
            f"{num_cells_sc_data}. Check that footprints and sc_data were loaded "
            f"from the same session."
        )
        
        # Precompute collapsed mixed cell sets once per session
        mixed_sets = {
            'partner_mixed': set(sc_data.get('partner_left_mixed', [])) | 
                             set(sc_data.get('partner_right_mixed', [])),
            'novel_mixed':   set(sc_data.get('novel_left_mixed', [])) | 
                             set(sc_data.get('novel_right_mixed', [])),
            'left_mixed':    set(sc_data.get('partner_left_mixed', [])) | 
                             set(sc_data.get('novel_left_mixed', [])),
            'right_mixed':   set(sc_data.get('partner_right_mixed', [])) | 
                             set(sc_data.get('novel_right_mixed', [])),
        }
        
        img_height, img_width = footprints.shape[1], footprints.shape[2]
        ax.set_xlim(0, img_width)
        ax.set_ylim(img_height, 0)
        ax.set_aspect('equal')
        
        for n in range(footprints.shape[0]):
            contour = get_neuron_contour(footprints[n], threshold=threshold)
            if contour is not None:
                fill_color = get_cell_color(n, sc_data, mixed_sets)
                ax.fill(contour[:, 1], contour[:, 0],
                        color=fill_color, alpha=fill_alpha, zorder=1)
                ax.plot(contour[:, 1], contour[:, 0],
                        color=outline_color, linewidth=outline_linewidth,
                        alpha=outline_alpha, zorder=2)
        
        ax.set_title(f'Cell tunings: {short_label}', fontsize=13,
                     color=outline_color, fontweight='bold')
        if background_color is not None:
            ax.set_facecolor(background_color)
        ax.axis('off')
    
    fig.suptitle('Cell tunings across sessions', fontsize=18,
                 color=outline_color, fontweight='bold')
    if background_color is not None:
        fig.set_facecolor(background_color)

    # Build legend handles - pure cells are always shown, mixed entries
    # depend on how mixed cells are being colored
    legend_handles = [
        Patch(facecolor=PARTNER_COLOR, edgecolor=outline_color,
              linewidth=outline_linewidth, label='Partner'),
        Patch(facecolor=NOVEL_COLOR, edgecolor=outline_color,
              linewidth=outline_linewidth, label='Novel'),
        Patch(facecolor=LEFT_COLOR, edgecolor=outline_color,
              linewidth=outline_linewidth, label='Left'),
        Patch(facecolor=RIGHT_COLOR, edgecolor=outline_color,
              linewidth=outline_linewidth, label='Right'),
    ]
    
    if color_mixed_by == 'identity':
        legend_handles.extend([
            Patch(facecolor=PARTNER_MIXED_COLOR, edgecolor=outline_color,
                  linewidth=outline_linewidth, label='Partner (mixed)'),
            Patch(facecolor=NOVEL_MIXED_COLOR, edgecolor=outline_color,
                  linewidth=outline_linewidth, label='Novel (mixed)'),
        ])
    elif color_mixed_by == 'location':
        legend_handles.extend([
            Patch(facecolor=LEFT_MIXED_COLOR, edgecolor=outline_color,
                  linewidth=outline_linewidth, label='Left (mixed)'),
            Patch(facecolor=RIGHT_MIXED_COLOR, edgecolor=outline_color,
                  linewidth=outline_linewidth, label='Right (mixed)'),
        ])
    
    legend_handles.append(
        Patch(facecolor='#D3D3D3', edgecolor=outline_color,
              linewidth=outline_linewidth, label='Untuned')
    )
    
    fig.legend(handles=legend_handles, loc='center left',
               bbox_to_anchor=(1.0, 0.5), frameon=False,
               labelcolor=outline_color, fontsize=11)
    
    plt.tight_layout()
    
    if save_path:
        plt.savefig(save_path, dpi=300, bbox_inches='tight')
    
    if show:
        plt.show()
    else:
        plt.close()
    
    return fig, axes

In [ ]:
#save out the following three:

In [ ]:
figsize_tuning_plots = (20,5)

In [ ]:
if savefigs:
    tuned_cells_8689_identity_mixed_path = os.path.join(cellreg_res_path,'8689_all_sessions_tuned_neurons_mixed_by_identity.svg')
else:
    tuned_cells_8689_identity_mixed_path = None

In [ ]:
animal = '8689'

fig, ax = color_footprints_by_tuning(animal, session_indices=None, footprints_list=footprints_lists_by_animal[animal],
                                 all_sc_data=all_sc_data, color_mixed_by='identity',
                                 threshold=0.07, figsize=figsize_tuning_plots, fill_alpha=0.8,
                                 outline_color='black', outline_linewidth=0.5,
                                 outline_alpha=0.8, background_color=None,
                                 show=True, save_path=tuned_cells_8689_identity_mixed_path)

In [ ]:
if savefigs:
    tuned_cells_8689_location_mixed_path = os.path.join(cellreg_res_path,'8689_all_sessions_tuned_neurons_mixed_by_location.svg')
else:
    tuned_cells_8689_location_mixed_path = None

In [ ]:
animal = '8689'

fig, ax = color_footprints_by_tuning(animal, session_indices=None, footprints_list=footprints_lists_by_animal[animal],
                                 all_sc_data=all_sc_data, color_mixed_by='location',
                                 threshold=0.07, figsize=figsize_tuning_plots, fill_alpha=0.8,
                                 outline_color='black', outline_linewidth=0.5,
                                 outline_alpha=0.8, background_color=None,
                                 show=True, save_path=tuned_cells_8689_location_mixed_path)

In [ ]:
if savefigs:
    tuned_cells_7981_identity_mixed_path = os.path.join(cellreg_res_path,'7981_all_sessions_tuned_neurons_mixed_by_identity.svg')
else:
    tuned_cells_7981_identity_mixed_path = None

In [ ]:
animal = '7981'

fig, ax = color_footprints_by_tuning(animal, session_indices=None, footprints_list=footprints_lists_by_animal[animal],
                                 all_sc_data=all_sc_data, color_mixed_by='identity',
                                 threshold=0.07, figsize=figsize_tuning_plots, fill_alpha=0.8,
                                 outline_color='black', outline_linewidth=0.5,
                                 outline_alpha=0.8, background_color=None,
                                 show=True, save_path=tuned_cells_7981_identity_mixed_path)

In [ ]:
if savefigs:
    tuned_cells_7981_location_mixed_path = os.path.join(cellreg_res_path,'7981_all_sessions_tuned_neurons_mixed_by_location.svg')
else:
    tuned_cells_7981_location_mixed_path = None

In [ ]:
animal = '7981'

fig, ax = color_footprints_by_tuning(animal, session_indices=None, footprints_list=footprints_lists_by_animal[animal],
                                 all_sc_data=all_sc_data, color_mixed_by='location',
                                 threshold=0.07, figsize=figsize_tuning_plots, fill_alpha=0.8,
                                 outline_color='black', outline_linewidth=0.5,
                                 outline_alpha=0.8, background_color=None,
                                 show=True, save_path=tuned_cells_7981_location_mixed_path)

In [ ]:
if savefigs:
    tuned_cells_8946_identity_mixed_path = os.path.join(cellreg_res_path,'8946_all_sessions_tuned_neurons_mixed_by_identity.svg')
else:
    tuned_cells_8946_identity_mixed_path = None

In [ ]:
animal = '8946'

fig, ax = color_footprints_by_tuning(animal, session_indices=None, footprints_list=footprints_lists_by_animal[animal],
                                 all_sc_data=all_sc_data, color_mixed_by='identity',
                                 threshold=0.07, figsize=figsize_tuning_plots, fill_alpha=0.8,
                                 outline_color='black', outline_linewidth=0.5,
                                 outline_alpha=0.8, background_color=None,
                                 show=True, save_path=tuned_cells_8946_identity_mixed_path)

In [ ]:
if savefigs:
    tuned_cells_8946_location_mixed_path = os.path.join(cellreg_res_path,'8946_all_sessions_tuned_neurons_mixed_by_location.svg')
else:
    tuned_cells_8946_location_mixed_path = None

In [ ]:
animal = '8946'

fig, ax = color_footprints_by_tuning(animal, session_indices=None, footprints_list=footprints_lists_by_animal[animal],
                                 all_sc_data=all_sc_data, color_mixed_by='location',
                                 threshold=0.07, figsize=figsize_tuning_plots, fill_alpha=0.8,
                                 outline_color='black', outline_linewidth=0.5,
                                 outline_alpha=0.8, background_color=None,
                                 show=True, save_path=tuned_cells_8946_location_mixed_path)

In [ ]:
#determining whether any cell tuning category is significantly stable relative to chance across coregistered cells in any pair of sessions

In [ ]:
def test_tuning_stability_session_pair(session_pair, all_sc_data, zero_index_maps,
                                         categories=None, n_shuffles=1000,
                                         percentile_cutoff=95, random_seed=None,
                                         verbose=True):
    """
    Test whether tuning stability across a session pair exceeds chance, pooled
    across animals while respecting animal boundaries in the shuffle.
    
    For each tuning category, the function:
      1. Identifies co-registered cells (present in both sessions) per animal
      2. Counts how many of those cells are in the category in session A AND 
         in session B ("true overlap")
      3. Shuffles tuning labels within each animal-session among co-registered 
         cells, preserving each session's category base rate, and recomputes 
         overlap. Repeats n_shuffles times to build a null distribution.
      4. Compares true overlap against the null distribution's upper percentile.
    
    Parameters
    ----------
    session_pair : tuple of 2 ints
        The two session indices to compare (e.g. (1, 2) for SS-N vs SS-F).
        Uses the same indexing as SESSION_SUBSETS['social'].
    all_sc_data : dict
        Nested dict: all_sc_data[animal_id][session_idx] -> single_cell_tuning dict.
    zero_index_maps : dict
        {animal_id: DataFrame} from CellReg loading code. DataFrame columns 
        should be session indices (post-relabeling).
    categories : list[str] or None
        Categories to test. Defaults to the 8 standard social categories.
    n_shuffles : int
        Number of shuffles for the null distribution. Default 1000.
    percentile_cutoff : float
        Upper percentile threshold for significance (default 95 for one-sided).
    random_seed : int or None
        For reproducibility.
    verbose : bool
        Print progress and per-category results.
    
    Returns
    -------
    pd.DataFrame
        One row per category with columns: category, n_coreg_total, 
        observed_overlap, null_mean, null_std, null_cutoff, p_value, significant,
        per_animal_contributions.
    """
    if random_seed is not None:
        np.random.seed(random_seed)

    social_categories = ['partner_cells', 'novel_cells', 'left_cells', 'right_cells',
                             'partner_left_mixed', 'partner_right_mixed',
                             'novel_left_mixed', 'novel_right_mixed']
    
    if categories is None:
        categories = social_categories
    
    s_a, s_b = session_pair
    
    # ---- Step 1: Build per-animal co-registered cell info ----
    # For each animal, we need:
    #   - list of (neuron_id_A, neuron_id_B) pairs for co-registered cells
    #   - for each category, the tuning status in session A and session B of those cells
    
    animal_coreg_data = {}
    
    for animal_id, zero_map in zero_index_maps.items():
        if s_a not in zero_map.columns or s_b not in zero_map.columns:
            continue
        
        # Co-registered cells: rows where both session entries are non-NaN
        both_present = zero_map[[s_a, s_b]].dropna()
        if len(both_present) == 0:
            print(f'skipping {animal_id}: no coregistered cells across sessions {s_a} and {s_b}')
            continue
        
        # Convert to native ints for indexing into category lists
        coreg_ids_a = both_present[s_a].astype(int).values
        coreg_ids_b = both_present[s_b].astype(int).values
        n_coreg = len(coreg_ids_a)
        
        # Get tuning dicts for each session; skip if either is missing
        sc_a = all_sc_data.get(animal_id, {}).get(s_a)
        sc_b = all_sc_data.get(animal_id, {}).get(s_b)
        if sc_a is None or sc_b is None:
            print(f'skipping {animal_id}: no single cell analysis results for session {s_a} and/or {s_b}')
            continue

        # Check that the session data actually contains social tuning keys
        # (not just that the session ran, but that the relevant analysis ran)
        if not all(cat in sc_a for cat in social_categories):
            if verbose:
                print(f"  Skipping {animal_id}: no social tuning results run for session {s_a}")
            continue
        if not all(cat in sc_b for cat in social_categories):
            if verbose:
                print(f"  Skipping {animal_id}: no social tuning results run for session {s_b}")
            continue
        
        # For each category, build boolean masks: is each co-registered cell 
        # in that category in session A? In session B?
        cat_masks_a = {}
        cat_masks_b = {}
        for cat in categories:
            set_a = set(sc_a.get(cat, []))
            set_b = set(sc_b.get(cat, []))
            cat_masks_a[cat] = np.array([nid in set_a for nid in coreg_ids_a])
            cat_masks_b[cat] = np.array([nid in set_b for nid in coreg_ids_b])
        
        animal_coreg_data[animal_id] = {
            'n_coreg': n_coreg,
            'cat_masks_a': cat_masks_a,
            'cat_masks_b': cat_masks_b,
        }
    
    if len(animal_coreg_data) == 0:
        print(f"No animals with data for session pair {session_pair}")
        return pd.DataFrame()
    
    # ---- Step 2: Compute observed overlaps ----
    observed_overlaps = {cat: 0 for cat in categories}
    per_animal_contributions = {cat: {} for cat in categories}
    n_coreg_total = 0
    
    for animal_id, data in animal_coreg_data.items():
        n_coreg_total += data['n_coreg']
        for cat in categories:
            # Overlap = cells tuned for cat in BOTH sessions
            overlap = int((data['cat_masks_a'][cat] & data['cat_masks_b'][cat]).sum())
            observed_overlaps[cat] += overlap
            per_animal_contributions[cat][animal_id] = overlap
    
    # ---- Step 3: Build null distribution via shuffling ----
    # For each shuffle iteration, shuffle each animal's session-B 
    # mask (breaking any cross-session correspondence while 
    # preserving the per-animal-session category base rate), then recompute 
    # pooled overlap.
    
    null_overlaps = {cat: np.zeros(n_shuffles, dtype=int) for cat in categories}
    
    for shuffle_idx in range(n_shuffles):
        if verbose and shuffle_idx % 200 == 0 and shuffle_idx > 0:
            print(f"  Shuffle {shuffle_idx}/{n_shuffles}")
        
        for animal_id, data in animal_coreg_data.items():
            n = data['n_coreg']
            # Generate one permutation of positions per session, per shuffle
            #wms edit - just permuting the second session, now, rather than both sessions as originally written
            #perm_a = np.random.permutation(n)
            perm_b = np.random.permutation(n)
            
            for cat in categories:
                shuffled_a = data['cat_masks_a'][cat]#[perm_a]
                shuffled_b = data['cat_masks_b'][cat][perm_b]
                null_overlaps[cat][shuffle_idx] += int((shuffled_a & shuffled_b).sum())
    
    # ---- Step 4: Compute p-values and summary stats ----
    rows = []
    for cat in categories:
        null_vals = null_overlaps[cat]
        obs = observed_overlaps[cat]
        null_mean = null_vals.mean()
        null_std = null_vals.std(ddof=1)
        null_cutoff = np.percentile(null_vals, percentile_cutoff)
        # One-sided empirical p-value (fraction of null ≥ observed)
        p_value = (null_vals >= obs).sum() / n_shuffles
        significant = obs > null_cutoff
        
        rows.append({
            'category': cat,
            'session_pair': f'{s_a}-{s_b}',
            'included_animals': [animal for animal in animal_coreg_data.keys()],
            'n_coreg_total': n_coreg_total,
            'observed_overlap': obs,
            'null_mean': null_mean,
            'null_std': null_std,
            'null_cutoff': null_cutoff,
            'p_value': p_value,
            'significant': significant,
            'per_animal_contributions': per_animal_contributions[cat],
        })
    
    results_df = pd.DataFrame(rows)
    
    if verbose:
        short_a = SESSION_CONFIG['short_labels'][s_a]
        short_b = SESSION_CONFIG['short_labels'][s_b]
        print(f"\nTuning stability: {short_a} ↔ {short_b}")
        print(f"Total co-registered cells across animals: {n_coreg_total}")
        print(f"{'Category':<25} {'Observed':>10} {'Null μ':>10} {'Null 95%':>10} {'p-value':>10}")
        print("-" * 70)
        for _, r in results_df.iterrows():
            sig_star = ' *' if r['significant'] else ''
            print(f"{r['category']:<25} {r['observed_overlap']:>10d} "
                  f"{r['null_mean']:>10.2f} {r['null_cutoff']:>10.1f} "
                  f"{r['p_value']:>10.4f}{sig_star}")
    
    return results_df

In [ ]:
from statsmodels.stats.multitest import multipletests

def correct_tuning_stability_p_vals(results_df,alpha=0.05):
    valid_pvals = results_df['p_value'].values
    rejected, p_corrected, _, _ = multipletests(valid_pvals, alpha=alpha, method='fdr_bh')
    results_df['sig_after_correction']=rejected
    results_df['p_fdr']=p_corrected
    results_df = results_df.rename(columns={'p_value':'p_uncorrected','significant':'sig_before_correction'})

    return results_df

In [ ]:
ss_sessions_tuning_stability = test_tuning_stability_session_pair(SESSION_SUBSETS['same_sex'], all_sc_data, zero_index_maps,
                                         categories=None, n_shuffles=10000,
                                         percentile_cutoff=95, random_seed=None,
                                         verbose=True)
ss_sessions_tuning_stability = correct_tuning_stability_p_vals(ss_sessions_tuning_stability)
ss_sessions_tuning_stability

In [ ]:
os_presep_sessions_tuning_stability = test_tuning_stability_session_pair(SESSION_SUBSETS['os_presep'], all_sc_data, zero_index_maps,
                                         categories=None, n_shuffles=10000,
                                         percentile_cutoff=95, random_seed=None,
                                         verbose=True)
os_presep_sessions_tuning_stability = correct_tuning_stability_p_vals(os_presep_sessions_tuning_stability)
os_presep_sessions_tuning_stability

In [ ]:
os_perisep_sessions_tuning_stability = test_tuning_stability_session_pair(SESSION_SUBSETS['os_familiar'], all_sc_data, zero_index_maps,
                                         categories=None, n_shuffles=10000,
                                         percentile_cutoff=95, random_seed=None,
                                         verbose=True)
os_perisep_sessions_tuning_stability = correct_tuning_stability_p_vals(os_perisep_sessions_tuning_stability)
os_perisep_sessions_tuning_stability

In [ ]:
novel_sessions_tuning_stability = test_tuning_stability_session_pair(SESSION_SUBSETS['novel'], all_sc_data, zero_index_maps,
                                         categories=None, n_shuffles=10000,
                                         percentile_cutoff=95, random_seed=None,
                                         verbose=True)
novel_sessions_tuning_stability = correct_tuning_stability_p_vals(novel_sessions_tuning_stability)
novel_sessions_tuning_stability

In [ ]:
familiar_sessions_tuning_stability = test_tuning_stability_session_pair(SESSION_SUBSETS['familiar_presep'], all_sc_data, zero_index_maps,
                                         categories=None, n_shuffles=10000,
                                         percentile_cutoff=95, random_seed=None,
                                         verbose=True)
familiar_sessions_tuning_stability = correct_tuning_stability_p_vals(familiar_sessions_tuning_stability)
familiar_sessions_tuning_stability

In [ ]:
all_stability_comparisons = {'ss_sessions_tuning_stability':ss_sessions_tuning_stability,
                             'os_presep_sessions_tuning_stability':os_presep_sessions_tuning_stability,
                             'os_perisep_sessions_tuning_stability':os_perisep_sessions_tuning_stability,
                             'novel_sessions_tuning_stability':novel_sessions_tuning_stability,
                             'familiar_sessions_tuning_stability':familiar_sessions_tuning_stability}

In [ ]:
cellreg_res_path

In [ ]:
if savecsvs:
    for name, df in all_stability_comparisons.items():
        df_save_path = os.path.join(cellreg_res_path,f'{name}_results.csv')
        df.to_csv(df_save_path,index=False)

In [ ]:
#plot the results of the stability calculation as a heatmap

In [ ]:
def plot_stability_heatmap_percent(all_results, categories=None, figsize=(12, 5),
                                     save_path=None, show=True):
    """
    Heatmap of observed overlap counts as percentage of total coregistered cells,
    with significance annotations.
    Rows = session pairs, columns = categories.
    """
    if categories is None:
        categories = ['partner_cells', 'novel_cells', 'left_cells', 'right_cells',
                      'partner_left_mixed', 'partner_right_mixed',
                      'novel_left_mixed', 'novel_right_mixed']
    
    session_pairs = list(all_results.keys())
    
    pct_matrix = np.full((len(session_pairs), len(categories)), np.nan)
    annot_matrix = [['' for _ in categories] for _ in session_pairs]
    
    for r, pair_name in enumerate(session_pairs):
        res_df = all_results[pair_name]
        if len(res_df) == 0:
            continue
        for c, cat in enumerate(categories):
            cat_row = res_df[res_df['category'] == cat]
            if len(cat_row) == 0:
                continue
            cat_row = cat_row.iloc[0]
            obs = cat_row['observed_overlap']
            n_coreg = cat_row['n_coreg_total']
            p_fdr = cat_row.get('p_fdr', np.nan)
            sig = cat_row.get('sig_after_correction', False)
            
            pct = (obs / n_coreg * 100) if n_coreg > 0 else 0
            pct_matrix[r, c] = pct
            
            star = '*' if sig else ''
            annot_matrix[r][c] = f'{pct:.1f}%{star}'
    
    fig, ax = plt.subplots(figsize=figsize)
    
    cat_labels = [c.replace('_cells', '').replace('_', '\n') for c in categories]
    
    sns.heatmap(pct_matrix, ax=ax, cmap='Blues', annot=annot_matrix, fmt='',
                xticklabels=cat_labels, yticklabels=session_pairs,
                linewidths=0.5, linecolor='white',
                cbar_kws={'label': '% of coregistered cells'})
    
    ax.set_title('Cross-session tuning stability\n(% of coregistered cells retaining tuning)',
                fontsize=12)
    ax.set_xlabel('Tuning category', fontsize=11)
    ax.set_ylabel('Session pair', fontsize=11)
    
    plt.tight_layout()
    
    if save_path:
        plt.savefig(save_path, dpi=300, bbox_inches='tight')
    if show:
        plt.show()
    else:
        plt.close()
    
    return fig, ax

In [ ]:
savefigs=True

In [ ]:
if savefigs:
    stable_tuning_types_heatmap_path = os.path.join(cellreg_res_path,'stability_of_tuning_types_across_session_pairs_heatmap.svg')
else:
    stable_tuning_types_heatmap_path = None

In [ ]:
fig, ax = plot_stability_heatmap_percent(all_stability_comparisons, categories=None, figsize=(12, 5),
                                     save_path=stable_tuning_types_heatmap_path, show=True)